# Sistema de recomendación de películas

Este notebook tiene dos partes:

1. **Recomendador basado en contenido** (la tarea principal): usa el dataset TMDB 5000 para
   recomendar películas parecidas a una dada. Genera `model.pkl` y `similarity.pkl`, que usa `app.py`.
2. **Bonus**: filtrado colaborativo basado en usuarios y un recomendador **híbrido**, con el
   dataset MovieLens (ml-latest-small), evaluados con métricas.

Los CSV de TMDB deben estar en `data/`. MovieLens se descarga automáticamente la primera vez.

In [1]:
import ast
import os
import pickle
import urllib.request
import zipfile

import numpy as np
import pandas as pd
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = 'data'

# Opciones del modelo de contenido.
# VECTORIZER puede ser 'tfidf' o 'count'. En la sección de evaluación se comparan ambos.
VECTORIZER = 'tfidf'
MAX_FEATURES = 5000   # tamaño del vocabulario (antes 500, demasiado pequeño)
N_RECS = 10
RANDOM_STATE = 42

## Parte 1 · Recomendador basado en contenido

### Paso 1: cargar y explorar los datos

In [2]:
movies_raw = pd.read_csv(os.path.join(DATA_DIR, 'tmdb_5000_movies.csv'))
credits = pd.read_csv(os.path.join(DATA_DIR, 'tmdb_5000_credits.csv'))
print('movies :', movies_raw.shape)
print('credits:', credits.shape)
movies_raw.head(2)

movies : (4803, 20)
credits: (4803, 4)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


**Cambio respecto a la versión original:** antes se unían las dos tablas por `title`.
Hay títulos repetidos que son películas distintas (por ejemplo *Batman* de 1966 y de 1989),
así que el merge por título cruzaba sus filas y creaba combinaciones falsas
(el reparto de una película con la sinopsis de la otra). Ahora se une por el **ID de TMDB**,
que es único: en `movies` se llama `id` y en `credits` se llama `movie_id`.

In [3]:
# credits también tiene una columna 'title'; la quitamos para no acabar con title_x / title_y
credits = credits.rename(columns={'movie_id': 'id'}).drop(columns='title')

# validate='one_to_one' hace que pandas lance un error si algún ID estuviera repetido
movies = movies_raw.merge(credits, on='id', how='inner', validate='one_to_one')
print('Tras el merge:', movies.shape)   # 4803 filas: ni una más, ni una menos

# Guardamos el año para distinguir títulos repetidos en la web: "Batman (1989)"
movies['year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year.astype('Int64')

movies = movies[['id', 'title', 'year', 'overview', 'keywords', 'genres', 'cast', 'crew']]
movies.isna().sum()

Tras el merge: (4803, 22)


id          0
title       0
year        1
overview    3
keywords    0
genres      0
cast        0
crew        0
dtype: int64

In [4]:
# Solo 'overview' tiene nulos (3 películas sin sinopsis): las quitamos
movies = movies.dropna(subset=['overview'])

# CAMBIO CLAVE: después de dropna el índice tiene huecos (…, 2657, 2659, …).
# La matriz de similitud se indexa por POSICIÓN (fila 0, 1, 2…), así que el índice
# del DataFrame tiene que coincidir con la posición. Sin este reset_index, casi la
# mitad de las películas recibían las recomendaciones de la película de al lado.
movies = movies.reset_index(drop=True)
print(movies.shape)

# Títulos repetidos que ahora son películas distintas (con su año):
movies[movies['title'].duplicated(keep=False)][['id', 'title', 'year']]

(4800, 8)


,id,title,year
972,72710,The Host,2013
1359,268,Batman,1989
2876,1255,The Host,2006
3646,39269,Out of the Blue,1980
3692,10844,Out of the Blue,2006
4265,2661,Batman,1966


### Paso 2: extraer las características y crear la columna `tags`

In [5]:
def extract_names(obj, limit=None):
    """Convierte el texto JSON de TMDB en una lista de nombres.
    Sirve para genres, keywords y cast (con limit=3 para los 3 actores principales)."""
    names = [item['name'] for item in ast.literal_eval(obj)]
    return names[:limit] if limit else names


def extract_director(obj):
    """Devuelve una lista con el director (o vacía si no aparece)."""
    for item in ast.literal_eval(obj):
        if item['job'] == 'Director':
            return [item['name']]
    return []


movies['overview'] = movies['overview'].apply(str.split)
movies['genres'] = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)
movies['cast'] = movies['cast'].apply(extract_names, limit=3)
movies['crew'] = movies['crew'].apply(extract_director)

# Guardamos una copia de los géneros para evaluar el modelo más adelante
movies['genre_list'] = movies['genres']
movies[['title', 'genres', 'cast', 'crew']].head()

,title,genres,cast,crew
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,Spectre,"[Action, Adventure, Crime]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,John Carter,"[Action, Adventure, Science Fiction]","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


**Cambio:** antes `tags` no incluía los géneros, aunque el comentario decía que sí.
Ahora se suman los cinco campos.

Además, a los nombres de varias palabras se les quitan los espacios (*Science Fiction* →
*ScienceFiction*, *Sam Worthington* → *SamWorthington*). Así el vectorizador los trata
como un único término y no confunde a *Sam Worthington* con cualquier otro *Sam*.

In [6]:
def collapse(names):
    return [name.replace(' ', '') for name in names]

movies['tags'] = (
    movies['overview']
    + movies['genres'].apply(collapse)     # <- NUEVO: los géneros ahora sí entran
    + movies['keywords'].apply(collapse)
    + movies['cast'].apply(collapse)
    + movies['crew'].apply(collapse)
)

# Stemming: reduce cada palabra a su raíz (loves/loved/loving -> love)
ps = PorterStemmer()
movies['tags'] = movies['tags'].apply(lambda words: ' '.join(ps.stem(w) for w in words))

movies.loc[0, 'tags'][:300]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplan'

### Paso 3: vectorizar y calcular la similitud del coseno

Se pueden usar dos vectorizadores:

- **CountVectorizer**: cuenta cuántas veces aparece cada palabra.
- **TF-IDF**: igual, pero da menos peso a las palabras que aparecen en muchísimas películas
  (*life*, *find*, *drama*…) y más a las que son características de pocas
  (*pirat*, *dinosaur*, *ChristopherNolan*…). Suele dar recomendaciones más específicas.

In [7]:
def build_similarity(tags, kind=VECTORIZER, max_features=MAX_FEATURES):
    if kind == 'tfidf':
        vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
    elif kind == 'count':
        vectorizer = CountVectorizer(max_features=max_features, stop_words='english')
    else:
        raise ValueError("kind debe ser 'tfidf' o 'count'")
    # Se deja la matriz dispersa (sin .toarray()): ocupa mucho menos y cosine_similarity la acepta
    vectors = vectorizer.fit_transform(tags)
    return cosine_similarity(vectors)

### Evaluación: ¿qué vectorizador funciona mejor?

No tenemos "respuestas correctas" para un recomendador de contenido, así que usamos una
métrica aproximada: para cada película, miramos sus 10 recomendaciones y calculamos cuánto se
parecen sus géneros (índice de **Jaccard**: géneros en común / géneros totales).
Como referencia se incluye un recomendador **aleatorio**.

Ojo: los géneros también forman parte de los `tags`, así que esta métrica **premia a los modelos
que dan mucho peso a los géneros**. Es justo lo que hace CountVectorizer (los géneros son
palabras muy frecuentes) y lo que TF-IDF evita a propósito. En la Parte 3 se compara con una
métrica externa (valoraciones de usuarios reales) que da el veredicto contrario.

In [8]:
genre_sets = [set(g) for g in movies['genre_list']]

def genre_jaccard_at_k(sim, k=N_RECS):
    sim = sim.copy()
    np.fill_diagonal(sim, -np.inf)                 # no recomendar la propia película
    top_k = np.argpartition(-sim, k, axis=1)[:, :k]
    scores = []
    for i, recs in enumerate(top_k):
        if not genre_sets[i]:
            continue
        scores.append(np.mean([
            len(genre_sets[i] & genre_sets[j]) / len(genre_sets[i] | genre_sets[j])
            for j in recs
        ]))
    return np.mean(scores)

rng = np.random.default_rng(RANDOM_STATE)
results = {'aleatorio (referencia)': genre_jaccard_at_k(rng.random((len(movies), len(movies))))}
for kind, max_features in [('count', 500), ('count', 5000), ('tfidf', 5000)]:
    sim = build_similarity(movies['tags'], kind, max_features)
    results[f'{kind}, {max_features} términos'] = genre_jaccard_at_k(sim)
    del sim

pd.Series(results, name='Jaccard de géneros @10').round(3).to_frame()

,Jaccard de géneros @10
aleatorio (referencia),0.170
"count, 500 términos",0.516
"count, 5000 términos",0.484
"tfidf, 5000 términos",0.358


In [9]:
# Modelo final con la configuración elegida arriba (VECTORIZER, MAX_FEATURES)
similarity = build_similarity(movies['tags'])
similarity.shape

(4800, 4800)

### Paso 4: probar las recomendaciones

In [10]:
def recommend(title, n=N_RECS, year=None):
    """Devuelve las n películas más parecidas a `title`.
    Si hay varias películas con ese título, se puede indicar el año."""
    matches = movies[movies['title'] == title]
    if year is not None:
        matches = matches[matches['year'] == year]
    if matches.empty:
        raise ValueError(f'No encuentro "{title}"')
    if len(matches) > 1:
        print(f'Aviso: hay {len(matches)} películas llamadas "{title}"; uso la primera. Indica year=…')

    pos = matches.index[0]            # tras reset_index, índice == posición en la matriz
    scores = similarity[pos]
    order = np.argsort(-scores)
    order = order[order != pos][:n]   # quitamos la propia película
    return movies.loc[order, ['title', 'year']].assign(similitud=scores[order].round(3))

recommend('Avatar')

,title,year,similitud
2403,Aliens,1986,0.252
3723,Falcon Rising,2014,0.226
582,Battle: Los Angeles,2011,0.194
1213,Aliens vs Predator: Requiem,2007,0.189
3603,Apollo 18,2011,0.177
47,Star Trek Into Darkness,2013,0.173
778,Meet Dave,2008,0.166
1201,Predators,2010,0.163
539,Titan A.E.,2000,0.157
942,The Book of Life,2014,0.155


In [11]:
recommend('The Dark Knight Rises')

,title,year,similitud
65,The Dark Knight,2008,0.461
428,Batman Returns,1992,0.408
299,Batman Forever,1995,0.334
119,Batman Begins,2005,0.324
1359,Batman,1989,0.291
210,Batman & Robin,1997,0.265
3853,"Batman: The Dark Knight Returns, Part 2",2013,0.242
9,Batman v Superman: Dawn of Justice,2016,0.231
2507,Slow Burn,2005,0.202
1181,JFK,1991,0.132


In [12]:
recommend('Batman', year=1989)

,title,year,similitud
210,Batman & Robin,1997,0.425
428,Batman Returns,1992,0.324
3,The Dark Knight Rises,2012,0.291
119,Batman Begins,2005,0.269
299,Batman Forever,1995,0.225
65,The Dark Knight,2008,0.208
9,Batman v Superman: Dawn of Justice,2016,0.200
813,Superman,1978,0.149
30,Spider-Man 2,2004,0.134
1469,Chill Factor,1999,0.131


### Paso 5: guardar el modelo

- `model.pkl`: solo las columnas que necesita la web (`id`, `title`, `year`).
- `similarity.pkl`: la matriz en **float32**. Ocupa la mitad que en float64 (~92 MB en lugar de
  ~185 MB), por debajo del límite de 100 MB por archivo de GitHub, y la precisión perdida
  no cambia el orden de las recomendaciones en la práctica.

In [13]:
with open('model.pkl', 'wb') as f:
    pickle.dump(movies[['id', 'title', 'year']], f)

with open('similarity.pkl', 'wb') as f:
    pickle.dump(similarity.astype(np.float32), f)

for name in ['model.pkl', 'similarity.pkl']:
    print(f'{name}: {os.path.getsize(name) / 1e6:.1f} MB')

model.pkl: 0.2 MB
similarity.pkl: 92.2 MB


---
## Parte 2 (bonus) · Filtrado colaborativo basado en usuarios

El filtrado colaborativo no mira el contenido de las películas, solo **quién valoró qué**.
Idea: para predecir cuánto le gustará a Ana una película, buscamos a los usuarios que
puntúan de forma parecida a Ana (sus "vecinos") y promediamos lo que ellos le dieron.

TMDB 5000 no tiene valoraciones individuales, así que usamos **MovieLens ml-latest-small**
(100 836 valoraciones de 610 usuarios). Su archivo `links.csv` incluye el ID de TMDB de cada
película, lo que nos permitirá combinarlo con el modelo de contenido en la parte híbrida.

In [14]:
ML_DIR = os.path.join(DATA_DIR, 'ml-latest-small')
if not os.path.exists(ML_DIR):
    zip_path = ML_DIR + '.zip'
    urllib.request.urlretrieve(
        'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)

ratings = pd.read_csv(os.path.join(ML_DIR, 'ratings.csv'))
links = pd.read_csv(os.path.join(ML_DIR, 'links.csv'))
ml_movies = pd.read_csv(os.path.join(ML_DIR, 'movies.csv'))
print(ratings.shape, 'usuarios:', ratings['userId'].nunique(), 'películas:', ratings['movieId'].nunique())
ratings.head()

(100836, 4) usuarios: 610 películas: 9724


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


### Separar entrenamiento y test

Para cada usuario apartamos el 20 % de sus valoraciones como **test**. El modelo solo ve el
80 % restante, y luego comprobamos si acierta las valoraciones que no ha visto.

In [15]:
test = ratings.groupby('userId').sample(frac=0.2, random_state=RANDOM_STATE)
train = ratings.drop(test.index)
print('train:', len(train), ' test:', len(test))

# Matriz usuario × película (0 = sin valorar)
user_ids = np.sort(ratings['userId'].unique())
item_ids = np.sort(ratings['movieId'].unique())
user_pos = pd.Series(np.arange(len(user_ids)), index=user_ids)
item_pos = pd.Series(np.arange(len(item_ids)), index=item_ids)

R = np.zeros((len(user_ids), len(item_ids)))
R[user_pos[train['userId']].to_numpy(), item_pos[train['movieId']].to_numpy()] = train['rating']
rated = R > 0
print('Matriz:', R.shape, f'- rellena al {rated.mean():.1%}')

train: 80672  test: 20164
Matriz: (610, 9724) - rellena al 1.4%


### Similitud entre usuarios

Cada usuario puntúa con su propia escala: para uno un 3 es "normal", para otro es "mala".
Por eso restamos a cada usuario su media antes de comparar (*centrado*). La similitud del
coseno sobre valoraciones centradas es prácticamente la **correlación de Pearson**.

In [16]:
user_mean = R.sum(axis=1) / rated.sum(axis=1)
R_centered = np.where(rated, R - user_mean[:, None], 0.0)

user_sim = cosine_similarity(R_centered)
np.fill_diagonal(user_sim, 0)   # un usuario no es vecino de sí mismo

### Predicción

Para el usuario *u* tomamos sus *k* vecinos más parecidos (con similitud positiva) y predecimos:

$$\hat r_{u,i} = \bar r_u + \frac{\sum_{v} \text{sim}(u,v)\,(r_{v,i} - \bar r_v)}{\sum_{v} \text{sim}(u,v) + \lambda}$$

donde la suma recorre los vecinos que han valorado la película *i*. El término λ
(*shrinkage*) evita que una película valorada por un solo vecino reciba una predicción extrema.

In [17]:
def predict_user_cf(u, k=30, shrinkage=1.0):
    """Predice la valoración del usuario u (posición) para TODAS las películas."""
    sims = user_sim[u]
    neighbours = np.argpartition(-sims, k)[:k]
    neighbours = neighbours[sims[neighbours] > 0]
    w = sims[neighbours]
    numerator = w @ R_centered[neighbours]
    denominator = w @ rated[neighbours]      # suma de similitudes de quienes la valoraron
    pred = user_mean[u] + numerator / (denominator + shrinkage)
    return np.clip(pred, 0.5, 5.0)

def predict_all(k, shrinkage=1.0):
    return np.vstack([predict_user_cf(u, k, shrinkage) for u in range(len(user_ids))])

### Evaluación: RMSE y MAE

Comparamos con dos referencias muy simples: predecir siempre la media global, y predecir
siempre la media del usuario. Un buen modelo colaborativo tiene que batirlas.

In [18]:
t_u = user_pos[test['userId']].to_numpy()
t_i = item_pos[test['movieId']].to_numpy()
y = test['rating'].to_numpy()

def rmse(pred): return np.sqrt(np.mean((pred - y) ** 2))
def mae(pred): return np.mean(np.abs(pred - y))

global_mean = train['rating'].mean()
rows = {
    'media global': (rmse(np.full_like(y, global_mean)), mae(np.full_like(y, global_mean))),
    'media del usuario': (rmse(user_mean[t_u]), mae(user_mean[t_u])),
}
for k in [10, 30, 60, 100]:
    P = predict_all(k)
    rows[f'user-based CF, k={k}'] = (rmse(P[t_u, t_i]), mae(P[t_u, t_i]))

pd.DataFrame(rows, index=['RMSE', 'MAE']).T.round(4)

,RMSE,MAE
media global,1.0498,0.8307
media del usuario,0.9518,0.7404
"user-based CF, k=10",0.9147,0.7039
"user-based CF, k=30",0.8973,0.6865
"user-based CF, k=60",0.8901,0.6793
"user-based CF, k=100",0.8868,0.6758


In [19]:
K_BEST = 60
P_cf = predict_all(K_BEST)

> **Alternativa con la librería `surprise`.** El enunciado la menciona (SVD, KNNBasic).
> No está en `requirements.txt` porque a menudo falla al instalarse con NumPy 2 y Python 3.12.
> Si quieres probarla, usa un entorno aparte con `numpy<2` y `pip install scikit-surprise`:
>
> ```python
> from surprise import Dataset, Reader, KNNBasic, SVD, accuracy
> from surprise.model_selection import train_test_split
> data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], Reader(rating_scale=(0.5, 5)))
> trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
> for algo in [KNNBasic(k=40, sim_options={'name': 'pearson', 'user_based': True}), SVD(random_state=42)]:
>     algo.fit(trainset)
>     accuracy.rmse(algo.test(testset))
> ```

---
## Parte 3 (bonus) · Recomendador híbrido

Combinamos los dos modelos para cada usuario:

- **Puntuación colaborativa**: la valoración predicha arriba.
- **Puntuación de contenido**: cuánto se parece cada película (según la matriz de similitud
  de TMDB) a las películas que el usuario puntuó con 4 o más en entrenamiento.

Ambas se normalizan a [0, 1] para cada usuario y se mezclan con un peso α:

$$\text{híbrido} = \alpha \cdot \text{colaborativo} + (1-\alpha) \cdot \text{contenido}$$

α = 1 es colaborativo puro y α = 0 es contenido puro. Solo se usan las películas que están
en los dos datasets (se enlazan mediante el ID de TMDB de `links.csv`).

In [20]:
# Enlazar cada película de MovieLens con su fila en la matriz de similitud de TMDB
tmdb_row = pd.Series(movies.index, index=movies['id'])
links = links.dropna(subset=['tmdbId']).astype({'tmdbId': int})
links['tmdb_row'] = links['tmdbId'].map(tmdb_row)
item_tmdb_row = links.set_index('movieId')['tmdb_row'].reindex(item_ids).to_numpy()

shared = np.where(~np.isnan(item_tmdb_row))[0]            # columnas de R que están en TMDB
shared_tmdb = item_tmdb_row[shared].astype(int)
content_sim = similarity[np.ix_(shared_tmdb, shared_tmdb)]  # similitud entre esas películas
print('Películas en ambos datasets:', len(shared))

Películas en ambos datasets: 3536


In [21]:
def minmax(x):
    span = x.max() - x.min()
    return (x - x.min()) / span if span > 0 else np.zeros_like(x)

def user_scores(u, csim=None):
    """Puntuaciones colaborativa y de contenido (normalizadas) sobre las películas compartidas.
    csim permite probar otra matriz de contenido; por defecto se usa content_sim."""
    csim = content_sim if csim is None else csim
    cf = minmax(P_cf[u, shared])
    liked = np.where(R[u, shared] >= 4)[0]             # lo que le gustó en entrenamiento
    content = csim[:, liked].mean(axis=1) if len(liked) else np.zeros(len(shared))
    return cf, minmax(content)

def hybrid_scores(u, alpha, csim=None):
    cf, content = user_scores(u, csim)
    return alpha * cf + (1 - alpha) * content

### Evaluación: precision@10 y recall@10

Ahora medimos si el sistema **recomienda** bien, no si predice la nota exacta. Para cada
usuario recomendamos 10 películas que no haya visto en entrenamiento y contamos cuántas
están entre las que puntuó con 4 o más en test.

- **precision@10**: de las 10 recomendadas, qué fracción le gustaron.
- **recall@10**: de las que le gustaron, qué fracción aparece en las 10.

Incluimos una referencia de **popularidad** (recomendar lo más valorado), que en MovieLens
suele ser sorprendentemente difícil de batir.

In [22]:
test_liked = test[test['rating'] >= 4]
relevant = {
    user_pos[u]: set(item_pos[g['movieId']].to_numpy())
    for u, g in test_liked.groupby('userId')
}
shared_set_pos = {c: n for n, c in enumerate(shared)}
popularity = minmax(rated[:, shared].sum(axis=0).astype(float))

def evaluate(score_fn, k=10):
    precisions, recalls = [], []
    for u, rel_items in relevant.items():
        rel = {shared_set_pos[i] for i in rel_items if i in shared_set_pos}
        if not rel:
            continue
        scores = score_fn(u).astype(float).copy()
        scores[rated[u, shared]] = -np.inf           # no recomendar lo ya visto
        top = np.argpartition(-scores, k)[:k]
        hits = len(rel.intersection(top))
        precisions.append(hits / k)
        recalls.append(hits / len(rel))
    return np.mean(precisions), np.mean(recalls)

rows = {'popularidad (referencia)': evaluate(lambda u: popularity)}
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    label = {0.0: 'contenido puro', 1.0: 'colaborativo puro'}.get(alpha, 'híbrido')
    rows[f'α = {alpha} ({label})'] = evaluate(lambda u, a=alpha: hybrid_scores(u, a))

pd.DataFrame(rows, index=['precision@10', 'recall@10']).T.round(4)

,precision@10,recall@10
popularidad (referencia),0.1143,0.1156
α = 0.0 (contenido puro),0.0236,0.0365
α = 0.25 (híbrido),0.0382,0.0592
α = 0.5 (híbrido),0.0971,0.1281
α = 0.75 (híbrido),0.1426,0.1568
α = 1.0 (colaborativo puro),0.1267,0.1353


### Volviendo a la pregunta: ¿TF-IDF o CountVectorizer?

Con MovieLens tenemos por fin una forma **externa** de juzgar el modelo de contenido: si sus
similitudes ayudan a acertar qué películas les gustan a usuarios reales. Repetimos la
evaluación con cada vectorizador (contenido puro y el híbrido con α = 0.75).

In [23]:
rows = {}
for kind, max_features in [('count', 500), ('count', 5000), ('tfidf', 5000)]:
    sim = build_similarity(movies['tags'], kind, max_features)
    csim = sim[np.ix_(shared_tmdb, shared_tmdb)]
    rows[f'{kind}, {max_features} términos'] = [
        evaluate(lambda u: hybrid_scores(u, 0.0, csim))[0],
        evaluate(lambda u: hybrid_scores(u, 0.75, csim))[0],
    ]
    del sim, csim

pd.DataFrame(rows, index=['precision@10 contenido puro', 'precision@10 híbrido α=0.75']).T.round(4)

,precision@10 contenido puro,precision@10 híbrido α=0.75
"count, 500 términos",0.0100,0.1192
"count, 5000 términos",0.0093,0.1207
"tfidf, 5000 términos",0.0236,0.1426


### Ejemplo: recomendaciones híbridas para un usuario

In [24]:
ALPHA = 0.5
ml_titles = ml_movies.set_index('movieId')['title']

def recommend_for_user(user_id, n=N_RECS, alpha=ALPHA):
    u = user_pos[user_id]
    scores = hybrid_scores(u, alpha)
    scores[rated[u, shared]] = -np.inf
    top = np.argsort(-scores)[:n]
    return pd.DataFrame({
        'título': ml_titles.loc[item_ids[shared[top]]].to_numpy(),
        'puntuación híbrida': scores[top].round(3),
    })

u = user_pos[1]
favoritas = np.argsort(-R[u])[:5]
print('Favoritas del usuario 1:', list(ml_titles.loc[item_ids[favoritas]]))
recommend_for_user(1)

Favoritas del usuario 1: ['Seven (a.k.a. Se7en) (1995)', 'Usual Suspects, The (1995)', 'X-Men (2000)', 'M*A*S*H (a.k.a. MASH) (1970)', 'Blazing Saddles (1974)']


,título,puntuación híbrida
0,"Fugitive, The (1993)",0.765
1,"Shawshank Redemption, The (1994)",0.749
2,Pulp Fiction (1994),0.748
3,Micmacs (Micmacs à tire-larigot) (2009),0.734
4,Brazil (1985),0.732
5,Star Wars: Episode IV - A New Hope (1977),0.731
6,"Great Escape, The (1963)",0.729
7,"Godfather, The (1972)",0.729
8,Star Wars: Episode VI - Return of the Jedi (1983),0.726
9,Dr. Strangelove or: How I Learned to Stop Worr...,0.719


### Conclusiones

- **Contenido.** Unir por ID, meter los géneros y resetear el índice arregla los errores de la
  versión original. Entre vectorizadores, la métrica de géneros prefiere CountVectorizer, pero la
  métrica con usuarios reales prefiere **TF-IDF** (más del doble de precisión en contenido puro),
  así que es la opción por defecto. Moraleja: una métrica mal elegida puede llevar a la decisión equivocada.
- **Colaborativo.** El filtrado basado en usuarios bate a las referencias en RMSE
  (≈0.89 frente a 0.95 de la media del usuario) y en precision@10 bate a la popularidad.
- **Contenido puro** recomienda peor que la popularidad: encuentra películas *parecidas*, pero no
  sabe cuáles son buenas ni cuáles ve la gente.
- **Híbrido.** Con α = 0.75 obtiene la mejor precision@10 y recall@10 de todos: el colaborativo
  aporta "qué gusta a gente como tú" y el contenido afina hacia el estilo del usuario.
  Con α bajo, el contenido domina y el resultado empeora.

Nota: los números dependen de la partición aleatoria (`RANDOM_STATE`); con otra semilla cambian
un poco, pero el orden entre modelos debería mantenerse.